#### Training GPT from Scratch on Discharge Summaries



In this lab, I will walk through a slightly modified version of this huggingface tutorial on training GPT from scratch: https://huggingface.co/learn/llm-course/en/chapter7/6

Additionally, I'll explore how training on a small corpus of discharge summaries affects the embeddings and generations of our model. I'll also primarily use the higher level "transformers" API. 

(For lower-level API, Professor Boussina recommends Andrej Karpathy's implementation of minGPT: https://github.com/karpathy/minGPT/tree/master)

This notebook can be run completely locally, but professor recommend using a GPU. If running in colab, connect to a T4 runtime.

In [1]:
!pip install transformers torch accelerate datasets

In [2]:
from transformers import AutoTokenizer, GPT2LMHeadModel, AutoConfig, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import torch.nn.functional as F
import torch
import math

/Users/emilychen/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(
/Users/emilychen/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:260: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  torch.utils._pytree._register_pytree_node(


In [ ]:
from google.colab import files

First, we'll load the data that we used in Lab 2. If using this Colab, upload the "lab2-data.csv" file. If working locally, point to the relevant directory.

In [ ]:
uploaded = files.upload()

Saving lab2-data.csv to lab2-data (1).csv


In [3]:
discharge_summaries = pd.read_csv('lab2-data.csv')
dataset = Dataset.from_pandas(discharge_summaries)
print(dataset)

Dataset({
    features: ['HADM_ID', 'TEXT'],
    num_rows: 3668
})


Previously, I have created my own vocabulary from the corpus. Here, I'll use a pre-trained tokenizer which unlike before, I can now compact any text, have a fixed vocabulary and eliminate unknown tokens. I'll also restrict the context length to 512, hyperparameter I can tune to control how much text the model can see at once. Finally, the tokenizer class, ```AutoTokenizer``` allows us to break our large discharge summaries into smaller chunks that fit into our context limit. 

In [4]:
context_length = 512
tokenizer = AutoTokenizer.from_pretrained("gpt2")

outputs = tokenizer(
    dataset[:10]["TEXT"],
    truncation=True,
    max_length=context_length,
    return_overflowing_tokens=True,
    return_length=True,
)

print(f"Input IDs length: {len(outputs['input_ids'])}")
print(f"Input chunk lengths: {(outputs['length'])}")
print(f"Chunk mapping: {outputs['overflow_to_sample_mapping']}")

/Users/emilychen/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input IDs length: 73
Input chunk lengths: [512, 512, 512, 512, 512, 512, 512, 512, 278, 512, 512, 512, 512, 512, 512, 512, 512, 512, 207, 512, 512, 512, 512, 512, 512, 512, 512, 315, 512, 110, 512, 512, 512, 512, 295, 512, 512, 512, 512, 512, 512, 512, 512, 475, 512, 512, 512, 512, 512, 115, 512, 512, 512, 512, 512, 512, 110, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 196, 512, 512, 512, 512, 437]
Chunk mapping: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9]


Each chunk will be list of indices corresponding to tokens in our vocabulary.

In [5]:
print(outputs['input_ids'][0])

[2782, 3411, 7536, 25, 220, 685, 1174, 17, 11623, 12, 17, 12, 24, 1174, 60, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 220, 3167, 10136, 7536, 25, 220, 220, 685, 1174, 17, 11623, 12, 17, 12, 1433, 1174, 60, 628, 198, 16177, 25, 26112, 2149, 8881, 198, 198, 3237, 6422, 444, 25, 198, 57, 420, 273, 1220, 406, 3798, 349, 198, 198, 8086, 1571, 33250, 1174, 37564, 4586, 6530, 1248, 3553, 1174, 60, 198, 23675, 20011, 2913, 25, 198, 45170, 2356, 198, 198, 24206, 311, 31839, 393, 10001, 17443, 34997, 25, 198, 30645, 8710, 516, 1627, 36075, 357, 3506, 5387, 45808, 934, 27208, 8, 198, 198, 18122, 286, 21662, 5821, 1108, 25, 198, 5246, 13, 685, 1174, 29870, 938, 3672, 1248, 3365, 1174, 60, 318, 281, 9508, 27406, 582, 351, 10768, 257, 419, 291, 45219, 5958, 357, 43435, 198, 49257, 9809, 287, 685, 1174, 17, 17464, 1174, 60, 351, 685, 1174, 14749, 357, 403, 8, 16003, 1174, 60, 352, 12067, 17, 11, 31312, 2579, 8085, 39, 70, 11, 10768, 198, 2781, 1373, 842, 3686, 3780, 11, 11607, 257, 

To see some of what those indices mean, we can convert the ids to tokens:

In [7]:
tokenizer.convert_ids_to_tokens([2782, 3411])

['Ad', 'mission']

We can also decode the whole list back to text as well:

In [6]:
tokenizer.decode(outputs['input_ids'][0])

'Admission Date:  [**2125-2-9**]              Discharge Date:   [**2125-2-16**]\n\n\nService: MEDICINE\n\nAllergies:\nZocor / Lescol\n\nAttending:[**Doctor Last Name 1857**]\nChief Complaint:\nChest pain\n\nMajor Surgical or Invasive Procedure:\nCentral venous line insertion (right internal jugular vein)\n\nHistory of Present Illness:\nMr. [**Known lastname 1858**] is an 84 yo man with moderate aortic stenosis (outside\nhospital echo in [**2124**] with [**Location (un) 109**] 1 cm2, gradient 28 mmHg, moderate\nmitral regurgitation, mild aortic insufficiency), chronic left\nventricular systolic heart failure with EF 25-30%, hypertension,\nhyperlipidemia, diabetes mellitus, CAD s/p CABG in [**2099**] with\nSVG-LAD-Diagonal, SVG-OM, and SVG-RPDA-RPL, with a re-do CABG in\n[**9-/2117**] with LIMA-LAD, SVG-OM, SVG-diagonal, and SVG-RCA. He also\nhas severe peripheral arterial disease s/p peripheral bypass\nsurgery. He presented to [**Hospital 1474**] Hospital ER this morning with\nshortness

Let's now apply the tokenizer across the whole dataset

In [8]:
def tokenize(element):
    outputs = tokenizer(
        element["TEXT"],
        truncation=True,
        max_length=context_length, # hyperparameter, 512 in our case
        # you don't want to keep the first 512 tokens of 
        # each note and throw the rest out, so set the below as True!
        # Nice to have: can experiment with stride=... so that you can have
        # the chunks overlap
        return_overflowing_tokens=True, 
        return_length=True,
    )
    return {"input_ids": outputs['input_ids']}


tokenized_dataset = dataset.map(
    tokenize, batched=True, remove_columns=dataset.column_names
)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Now we'll load in a randomly initialized model with the GPT2 architecture:

In [ ]:
config = AutoConfig.from_pretrained(
    "gpt2",
    vocab_size=len(tokenizer),
    n_ctx=context_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    n_layer=6,
    n_head=6
)
model = GPT2LMHeadModel(config)

Let's explore this architecture a bit further:

In [ ]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

##### Q1 - What do WTE and WPE refer to?

Similar to Lab 2, we can can search for similar tokens using vector distances in the embedding space.

In [ ]:
tokens = tokenizer.encode('hospital')
result_vector = model.transformer.wte.weight[tokens].mean(axis=0)

similarities = F.cosine_similarity(
    result_vector,
    model.transformer.wte.weight
)
top_indices = similarities.topk(10).indices
print([tokenizer.decode(idx) for idx in top_indices if idx not in tokens])

[' Designer', 'tmp', ' Society', ' Girl', ' KR', ' Chak', '880', ' Editorial', ' serial']


#### Q2 - What do you observe about these tokens? Why?

##### Q3 - Use this model to generate some text. Describe the output.
(you may use model.generate(max_length=100))

In [ ]:
# YOUR CODE HERE

##### Forward Pass

Let's pass one of our input vectors in and explore the outputs.

First, we'll pass our tokenized input and retrieve the embeddings.

In [ ]:
tokenized_input = torch.tensor(outputs['input_ids'][0:1])
token_embeddings = model.transformer.wte(tokenized_input)
print(token_embeddings)

tensor([[[ 5.9578e-03,  1.7905e-02, -1.6835e-02,  ...,  2.1372e-03,
           2.6071e-02,  9.7751e-03],
         [ 6.7577e-03, -1.7092e-02, -1.0215e-02,  ..., -2.1808e-04,
          -1.7656e-02, -3.8405e-03],
         [-7.4139e-04, -6.6802e-03,  3.0292e-02,  ..., -4.5449e-05,
          -2.6317e-02,  1.2559e-02],
         ...,
         [-3.3299e-02, -6.5420e-03,  6.4523e-03,  ...,  1.5434e-02,
          -1.9761e-02,  5.0314e-03],
         [-1.4368e-02, -3.9883e-02, -5.0204e-02,  ...,  5.3913e-02,
           1.8680e-02,  8.4966e-03],
         [-1.2039e-02, -1.2343e-02,  2.5679e-02,  ..., -8.7259e-03,
          -1.1210e-02, -2.1614e-02]]], grad_fn=<EmbeddingBackward0>)


##### Q4 - What is the size of the embedding space?

In [ ]:
# YOUR CODE HERE

##### Q5 - Generate the position embeddings

In [ ]:
# YOUR CODE HERE

##### Q6 - Combine the position and token embeddings

In [ ]:
# YOUR CODE HERE
# hidden_states =

Now we'll layer normalize. Our transformer has 6 attention heads, let's dive into one of them:

In [ ]:
normalized_hidden_states = model.transformer.h[0].ln_1(hidden_states)

print("Unnormalized:")
print(hidden_states)
print('-----------------')
print("Normalized:")
print(normalized_hidden_states)

Unnormalized:
tensor([[[ 1.6187e-02, -1.4326e-02,  5.7182e-03,  ...,  1.0908e-02,
           3.6664e-02,  4.7981e-02],
         [ 6.3361e-03, -3.4105e-02, -2.7012e-02,  ..., -2.1553e-03,
          -1.8757e-03,  9.3575e-03],
         [ 1.0974e-02, -1.6824e-02, -2.0724e-03,  ..., -9.0751e-03,
          -6.9441e-05,  1.9950e-02],
         ...,
         [-1.5874e-02, -1.4800e-02,  5.2156e-03,  ...,  5.5615e-02,
          -3.1297e-02, -1.2484e-02],
         [-3.6596e-02, -7.4518e-02, -8.6467e-02,  ...,  7.2402e-02,
          -9.4106e-03,  2.7839e-02],
         [-1.3103e-02, -2.1964e-02,  7.7257e-03,  ..., -9.2802e-03,
          -1.2048e-02,  2.0359e-02]]], grad_fn=<AddBackward0>)
-----------------
Normalized:
tensor([[[ 0.5232, -0.5543,  0.1535,  ...,  0.3368,  1.2463,  1.6459],
         [ 0.2164, -1.1996, -0.9513,  ..., -0.0810, -0.0712,  0.3221],
         [ 0.3755, -0.5907, -0.0780,  ..., -0.3214, -0.0084,  0.6874],
         ...,
         [-0.6145, -0.5771,  0.1205,  ...,  1.8770, -1.1521

Let's take a look at the self-attention matrices

In [ ]:
model.transformer.h[0].attn.c_attn.weight.shape

torch.Size([768, 2304])

In class, we discussed the Wq, Wk, and Wv matrices. In practice, these are often stacked into a single matrix for efficient computation. So the matrix above represents [Wq Wk Wv].

We can derive our Q, K, V matrices by splitting the output:

In [ ]:
Q, K, V = model.transformer.h[0].attn.c_attn(normalized_hidden_states).split(768, dim=2)
print('Q:', Q)
print('K:', K)
print('V:', V)

Q: tensor([[[ 0.0480, -0.8048,  0.0761,  ...,  0.4841, -1.0344, -0.6923],
         [-0.0825,  0.3869, -0.8377,  ..., -0.0207,  0.2524, -0.5831],
         [ 0.1529, -0.5832,  0.8877,  ...,  0.2412, -0.3103,  0.0278],
         ...,
         [-0.4621,  0.3452,  0.1760,  ...,  0.1751,  1.2720,  0.2419],
         [ 0.9234,  0.0668,  0.6759,  ...,  0.2996, -0.6901, -0.4856],
         [-0.0583,  0.4008, -0.3541,  ..., -1.1880,  0.0489,  1.2265]]],
       grad_fn=<SplitBackward0>)
K: tensor([[[ 9.2760e-02,  4.8735e-01, -1.4685e-01,  ...,  1.1932e+00,
           3.0719e-01, -1.3298e+00],
         [-4.3925e-01,  5.0488e-01,  4.3878e-01,  ...,  1.8786e-01,
          -8.6731e-01,  2.4887e-02],
         [ 5.7886e-01,  1.5765e-04,  2.3221e-01,  ..., -1.1220e-01,
           2.7275e-01, -1.3518e-01],
         ...,
         [-7.0419e-01,  2.9667e-01,  6.8457e-01,  ...,  5.6274e-01,
          -3.2815e-01,  1.4125e+00],
         [ 5.0120e-02,  7.8891e-01, -8.9342e-01,  ...,  5.1636e-01,
           2.4565

In [ ]:
print(Q.shape)

torch.Size([1, 512, 768])


Remember, there's nothing special about these matrices/vectors. They are random. They only gain significance during training because the following constraint is applied during the forward pass:

In [ ]:
att = (Q @ K.transpose(-2, -1)) * (1.0 / math.sqrt(K.size(-1)))
print("QK^T Dim:", att.shape)
A = F.softmax(att, dim=-1)
Z = A @ V
print("Z Dim:", Z.shape)

QK^T Dim: torch.Size([1, 512, 512])
Z Dim: torch.Size([1, 512, 768])


Finally, the output of our model will be a matrix of logits (dimensionality of our vocabulary) for each position in the sequence. During training, we will compute the cross entropy between that logit and the embedding vectors of the next tokens (i.e. shifted by 1).

In [ ]:
model(tokenized_input).logits.shape

torch.Size([1, 512, 50257])

##### Model Training

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
args = TrainingArguments(
    output_dir="lab3",
    per_device_train_batch_size=14, # increase/decrease this based on your memory
    eval_steps=50,
    logging_steps=50,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    weight_decay=0.1,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    save_steps=500,
    fp16=True,
    report_to="none"
    # use_cpu=True # Very slow! Feel free to use without a GPU if you'd like
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset
)

The model will internally handle the next-token prediction loss. Go ahead and start the training. This will take a while!

In [ ]:
trainer.train()

##### Q7 - What are the trained embeddings most similar to "hospital"? How do they compare to the tokens from Q2?

In [ ]:
# YOUR CODE HERE

##### Q8 - Use the trained model to generate  a few samples of text. How does it compare to the text generated in Q3?

In [ ]:
# YOUR CODE HERE
# model.generate(max_length=100, do_sample=True, temperature=0.1)